# Platelet R2 — VALIDATED SAFE launcher v3

Self-contained launcher for the corrected **49-cell analysis notebook**. Execute this
launcher only in Google Colab after confirming the original Excel is available under
`MyDrive/Platelets/`. It has its own verified copy of the corrected notebook, never
uses the old `Platelet_Manuscript_Reproducibility.ipynb` path, and saves **every run**
under a unique timestamp/ID folder. No older results are overwritten.

The standalone analysis notebook in the ZIP is also executable directly and is easier
for interactive debugging.


In [3]:
# SAFE self-contained Colab launcher for the corrected 49-cell R2 v3 notebook.
# Does not load, overwrite, or trust the old 2-cell original-named notebook.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import base64, gzip, hashlib, json, os, shutil, subprocess, sys, uuid
try:
    import nbformat
    from nbclient import NotebookClient
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','nbformat>=5','nbclient>=0.9','ipykernel'])
    import nbformat
    from nbclient import NotebookClient

BASE = Path('/content/drive/MyDrive/Platelets')
DATA = BASE / 'platelet_data_english_260529.xlsx'
if not DATA.is_file():
    raise FileNotFoundError(f'Missing input Excel: {DATA}')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
WORKDIR = Path('/content/platelet_r2_validated_runs') / RUN_ID
ARCHIVE = BASE / 'Reproducibility_Results' / 'R2_VALIDATED_V3_RUNS' / RUN_ID
if WORKDIR.exists() or ARCHIVE.exists():
    raise FileExistsError('Generated run ID unexpectedly exists; rerun launcher to obtain another ID')
WORKDIR.mkdir(parents=True)
ARCHIVE.mkdir(parents=True)
shutil.copy2(DATA,WORKDIR/DATA.name)
EMBEDDED_GZIP_BASE64 = '...'
EXPECTED_NOTEBOOK_SHA256 = 'df2dc7508600d551ba643e20c603c680bc67fc3fa142787e46f09f14299f24a8'
nb_bytes=gzip.decompress(base64.b64decode(EMBEDDED_GZIP_BASE64))
assert hashlib.sha256(nb_bytes).hexdigest()==EXPECTED_NOTEBOOK_SHA256
notebook=nbformat.reads(nb_bytes.decode('utf-8'),as_version=4)
nbformat.validate(notebook)
assert len(notebook.cells)==49 and sum(c.cell_type=='code' for c in notebook.cells)==25
assert 'source_date = origin_date + pd.Timedelta(days=horizon - cycles_back * seasonal_period)' in notebook.cells[7].source
assert 'initial_reference_demand' in notebook.cells[7].source
# Outer launcher owns Drive authorization and unique archive. Inner kernel must NOT mount or archive.
notebook.cells[3].source = (
    'import os, sys, json, math, importlib.util, pathlib\n'
    'IN_COLAB = True\n'
    'RUN_ID = '+repr(RUN_ID)+'\n'
    'WORKDIR = pathlib.Path('+repr(str(WORKDIR))+')\n'
    'os.chdir(WORKDIR)\n'
    'print("Unique R2 work directory:", WORKDIR)'
)
notebook.cells[5].source = (
    'DATA_FILENAME = "platelet_data_english_260529.xlsx"\n'
    'from pathlib import Path\n'
    'assert Path(DATA_FILENAME).is_file()\n'
    'print("Input Excel:",Path(DATA_FILENAME).resolve())'
)
notebook.cells[48].source = '# Drive archiving is handled by the outer safe launcher (unique RUN_ID).\nprint("Validated analysis finished. Unique archive handled by launcher.")'

# Preserve exact corrected source and the actual environment-adapted execution notebook.
source_path = ARCHIVE / 'Platelet_R2_Validated_Integrated_v3_SOURCE.ipynb'
source_path.write_bytes(nb_bytes)
executed_path = ARCHIVE / 'Platelet_R2_Validated_Integrated_v3_EXECUTED.ipynb'

def on_cell_start(cell, cell_index):
    if cell.cell_type=='code':print(f'Executing code cell {cell_index+1}/49',flush=True)

client=NotebookClient(notebook,timeout=1800,kernel_name='python3',
    resources={'metadata': {'path': '/content'}}, on_cell_start=on_cell_start)
success=False
try:
    client.execute()
    success=True
finally:
    nbformat.write(notebook,str(executed_path))
    for folder in ('analysis_r2_s18s19','tables_r2_s18s19','figures_r2_s18s19'):
        src=WORKDIR/folder
        if src.is_dir():shutil.copytree(src,ARCHIVE/folder)
    manifest={'run_id':RUN_ID,'source_sha256':EXPECTED_NOTEBOOK_SHA256,
        'input_sha256':hashlib.sha256(DATA.read_bytes()).hexdigest(),
        'status':'COMPLETED' if success else 'FAILED',
        'workdir':str(WORKDIR),'python':sys.version.split()[0]}
    (ARCHIVE/'launcher_run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Executed notebook:',executed_path)
    print('Results:',ARCHIVE)
    print('Status:',manifest['status'])
print('All 49 cells executed successfully.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Executing code cell 3/49
Executing code cell 4/49
Executing code cell 6/49
Executing code cell 8/49
Executing code cell 9/49
Executing code cell 11/49
Executing code cell 13/49
Executing code cell 14/49
Executing code cell 16/49
Executing code cell 18/49
Executing code cell 21/49
Executing code cell 23/49
Executing code cell 25/49
Executing code cell 28/49
Executing code cell 29/49
Executing code cell 31/49
Executing code cell 33/49
Executing code cell 35/49
Executing code cell 37/49
Executing code cell 39/49
Executing code cell 40/49
Executing code cell 42/49
Executing code cell 44/49
Executing code cell 46/49
Executing code cell 49/49
Executed notebook: /content/drive/MyDrive/Platelets/Reproducibility_Results/R2_VALIDATED_V3_RUNS/20261117T071604Z_be04aacd/Platelet_R2_Validated_Integrated_v3_EXECUTED.ipynb
Results: /content/drive/MyDrive/Platelets/Reproducib